In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate momentum, efficiency, relative performance, and factor exposure for a single asset.
#Original Risk Analysis blocks included here: 14-21.


In [ ]:
qp = Plotter()
qe = MacroDataClient()
helper = Helper()
model = Models()
lineChartPlotter = LineChartPlotter()
candleStickPlotter = CandleStickPlotter()
barChartPlotter = BarChartPlotter()

PARAMS_FILE_CANDIDATES = []
for base_path in (Path.cwd(), *Path.cwd().parents):
    PARAMS_FILE_CANDIDATES.extend([
        base_path / "params.py",
        base_path / "Notebooks" / "Single Asset Profile" / "params.py",
    ])

seen_params_paths = set()
for params_file in PARAMS_FILE_CANDIDATES:
    params_file = params_file.resolve()
    if params_file in seen_params_paths:
        continue
    seen_params_paths.add(params_file)
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
get_single_asset_profile_params = notebook_params["get_single_asset_profile_params"]
resolve_time_frame_map = notebook_params["resolve_time_frame_map"]


PLOTLY_NOTEBOOK_CONFIG = {"responsive": True, "scrollZoom": True}
PLOTLY_DARK_THEME = {
    "template": "plotly_dark",
    "paper_bgcolor": "#05070b",
    "plot_bgcolor": "#05070b",
    "font": dict(color="#e5e7eb"),
    "legend": dict(
        bgcolor="rgba(5, 7, 11, 0.55)",
        bordercolor="rgba(148, 163, 184, 0.20)",
        borderwidth=1,
    ),
    "hoverlabel": dict(
        bgcolor="#0f172a",
        bordercolor="#1f2937",
        font=dict(color="#f8fafc"),
    ),
}
PLOTLY_DARK_GRID = "rgba(148, 163, 184, 0.16)"
PLOTLY_DARK_AXIS = "rgba(148, 163, 184, 0.24)"
PLOTLY_DARK_TEXT = "rgba(226, 232, 240, 0.92)"
PLOTLY_DARK_MENU_BG = "rgba(9, 14, 24, 0.95)"

for renderer_name in ("plotly_mimetype", "notebook", "notebook_connected", "jupyterlab"):
    try:
        pio.renderers[renderer_name].config = PLOTLY_NOTEBOOK_CONFIG.copy()
    except Exception:
        pass

def _coerce_dark_foreground(color):
    if not isinstance(color, str):
        return color
    normalized = color.replace(" ", "").lower()
    if normalized in {"black", "#000", "#000000", "rgb(0,0,0)", "rgba(0,0,0,0.85)", "rgba(0,0,0,1)"}:
        return "rgba(241, 245, 249, 0.88)"
    return color

def apply_plotly_dark_theme(fig):
    fig.update_layout(**PLOTLY_DARK_THEME)
    fig.update_xaxes(
        showgrid=True,
        gridcolor=PLOTLY_DARK_GRID,
        zeroline=False,
        showline=True,
        linecolor=PLOTLY_DARK_AXIS,
    )
    fig.update_yaxes(
        showgrid=True,
        gridcolor=PLOTLY_DARK_GRID,
        zeroline=False,
        showline=True,
        linecolor=PLOTLY_DARK_AXIS,
    )
    fig.update_scenes(
        bgcolor="#05070b",
        xaxis=dict(showbackground=True, backgroundcolor="#05070b", gridcolor=PLOTLY_DARK_GRID, zerolinecolor=PLOTLY_DARK_AXIS),
        yaxis=dict(showbackground=True, backgroundcolor="#05070b", gridcolor=PLOTLY_DARK_GRID, zerolinecolor=PLOTLY_DARK_AXIS),
        zaxis=dict(showbackground=True, backgroundcolor="#05070b", gridcolor=PLOTLY_DARK_GRID, zerolinecolor=PLOTLY_DARK_AXIS),
    )

    for trace in fig.data:
        trace_line = getattr(trace, "line", None)
        trace_line_color = getattr(trace_line, "color", None) if trace_line is not None else None
        if trace_line_color is not None:
            trace.line.color = _coerce_dark_foreground(trace_line_color)

    for shape in fig.layout.shapes or []:
        shape_line = getattr(shape, "line", None)
        shape_line_color = getattr(shape_line, "color", None) if shape_line is not None else None
        if shape_line_color is not None:
            shape.line.color = _coerce_dark_foreground(shape_line_color)

    for annotation in fig.layout.annotations or []:
        font_payload = annotation.font.to_plotly_json() if annotation.font else {}
        current_color = font_payload.get("color")
        if current_color is None:
            font_payload["color"] = PLOTLY_DARK_TEXT
        else:
            font_payload["color"] = _coerce_dark_foreground(current_color)
        annotation.font = font_payload

    for menu in fig.layout.updatemenus or []:
        menu.bgcolor = PLOTLY_DARK_MENU_BG
        menu.bordercolor = PLOTLY_DARK_AXIS
        menu.borderwidth = 1
        menu.font = dict(color="#e5e7eb", size=12)

    return fig

def show_plotly_figure(fig, *, config=None, **layout_kwargs):
    merged_config = PLOTLY_NOTEBOOK_CONFIG.copy()
    if config:
        merged_config.update(config)
    fig.update_layout(autosize=True, **layout_kwargs)
    apply_plotly_dark_theme(fig)
    fig.show(config=merged_config)

In [ ]:
# Block 3: load shared analysis parameters
# Edit Notebooks/Single Asset Profile/params.py to update these defaults for every notebook.
single_asset_params = get_single_asset_profile_params()

interval = single_asset_params["interval"]

In [ ]:
# Block 4: organize structural, position, swing timeframe lists

strategy = str(trading_strategy).strip().lower()
time_frame_map = resolve_time_frame_map(strategy)
time_frame_short = time_frame_map["short"]
time_frame_mid = time_frame_map["mid"]
time_frame_long = time_frame_map["long"]

return_frequencies = ('monthly', 'weekly', 'daily')


In [ ]:
# Block 5: load and align asset, benchmark, and return data

# Download and normalize asset-level data
ticker = yf.Ticker(ticker_str).history(period=period, interval=interval)
vix = yf.Ticker('^VIX').history(period=period, interval=interval)
risk_free_proxy = yf.Ticker(risk_free_ticker).history(period=period, interval=interval)
ticker = helper.simplify_datetime_index(ticker)
vix = helper.simplify_datetime_index(vix)
risk_free_proxy = helper.simplify_datetime_index(risk_free_proxy)
if risk_free_proxy.empty or 'Close' not in risk_free_proxy:
    raise ValueError(f"No risk-free history available for {risk_free_ticker}.")
risk_free_annual_yield = risk_free_proxy['Close'].dropna().sort_index().div(100)
risk_free_daily_rate = ((1 + risk_free_annual_yield) ** (1 / 252) - 1).shift(1)

# Download benchmark data once and keep it in collections for downstream cells
benchmark_tickers = normalize_benchmark_tickers(benchmark_tickers, ticker_str, include_asset=True)
benchmark_data, skipped_benchmarks = load_benchmark_data(benchmark_tickers, period, interval, helper)
if skipped_benchmarks:
    print(f'Skipped benchmarks with no data: {skipped_benchmarks}')

analysis_index, ticker, vix, benchmark_data = align_series_to_common_index(ticker, vix, benchmark_data)
risk_free_daily_rate = risk_free_daily_rate.reindex(ticker.index).ffill()

# Calculate asset and benchmark returns for the frequencies used elsewhere in the notebook
ticker_returns = {frequency: series_transforms.returns(ticker, frequency=frequency) for frequency in return_frequencies}
ticker_monthly_returns = ticker_returns['monthly']
ticker_weekly_returns = ticker_returns['weekly']
ticker_daily_returns = ticker_returns['daily']

benchmark_returns = {
    symbol: {frequency: series_transforms.returns(frame, frequency=frequency) for frequency in return_frequencies}
    for symbol, frame in benchmark_data.items()
}

vix_returns = {frequency: series_transforms.returns(vix, frequency=frequency) for frequency in return_frequencies}
vix_monthly_returns = vix_returns['monthly']
vix_weekly_returns = vix_returns['weekly']
vix_daily_returns = vix_returns['daily']


In [ ]:
# Block 6: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX 

ticker_vix_fix = rolling.vix_fix(ticker)
benchmark_vix_fix = {symbol: rolling.vix_fix(frame) for symbol, frame in benchmark_data.items()}

fig = qp.plot_series_with_stdev_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3],
    show=False,
)
show_plotly_figure(fig)


In [ ]:
# Block 7: stack candlestick, drawdown comparison, and rolling recovery time

peak_damage_window_options = [21, 50, 200]

peak_damage_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=ticker['Close'],
    windows=peak_damage_window_options,
    default_window=200 if 200 in peak_damage_window_options else max(peak_damage_window_options),
)

fig = lineChartPlotter.plot_candlestick_drawdown_recovery_profile(
    price_frame=ticker,
    metrics_by_window=peak_damage_context['metrics_by_window'],
    window_options=peak_damage_context['windows'],
    default_window=peak_damage_context['default_window'],
    ticker_label=ticker_str,
    candlestick_period=period,
    default_timeframe_label='10 Years',
)
show_plotly_figure(fig)


In [ ]:
# Block 8: compute rolling Sharpe windows, momentum histograms, and volatility

window_sizes = list(range(3, 201))

momentum_diagnostics_context = momentum_analytics.build_momentum_window_diagnostics_context(
    close_series=ticker['Close'],
    window_sizes=window_sizes,
    highlight_windows=(7, 21, 50, 200),
    surface_years=10,
    risk_free_rate=risk_free_daily_rate,
)

momentum_diagnostic_figs = lineChartPlotter.plot_momentum_window_diagnostics(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

show_plotly_figure(momentum_diagnostic_figs['optimal_window'])
show_plotly_figure(momentum_diagnostic_figs['optimal_window_histogram'])
show_plotly_figure(momentum_diagnostic_figs['sharpe_mean_median'])
show_plotly_figure(momentum_diagnostic_figs['volatility_mean_median'])
show_plotly_figure(momentum_diagnostic_figs['sharpe_surface_3d'])


In [ ]:
# Block 12b: plot stacked rolling Sharpe z-score heatmaps for 1-200 day windows

heatmap_windows = list(range(1, 201))
heatmap_time_frame_map = {f"window_{window}": window for window in heatmap_windows}

sharpe_heatmap_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=ticker["Close"],
    time_frame_map=heatmap_time_frame_map,
    benchmark_data=benchmark_data,
    risk_free_rate=risk_free_daily_rate,
)

heatmap_term_config_map = sharpe_heatmap_context["term_config_map"]
sharpe_zscore_series_by_window = {
    config["time_frame"]: config["sharpe_zscore"]
    for config in heatmap_term_config_map.values()
}
sharpe_zscore_heatmap = pd.DataFrame(sharpe_zscore_series_by_window).sort_index()
heatmap_matrix = sharpe_zscore_heatmap.T.reindex(heatmap_windows)
asset_daily_mean = heatmap_matrix.mean(axis=0, skipna=True).dropna().sort_index()

benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
    asset_sharpe_map=sharpe_heatmap_context["asset_sharpe_map"],
    asset_component_map=sharpe_heatmap_context["asset_component_map"],
    benchmark_metrics=sharpe_heatmap_context["benchmark_metrics"],
    spread_plot_data=sharpe_heatmap_context["spread_plot_data"],
    time_frame_map=heatmap_time_frame_map,
)

benchmark_heatmap_matrices = {}
benchmark_daily_mean = {}
for symbol in benchmark_plot_payload["benchmark_order"]:
    symbol_series_by_window = {
        window: benchmark_plot_payload["summary_zscore_map"].get(term_key, {}).get(symbol, pd.Series(dtype=float))
        for term_key, window in heatmap_time_frame_map.items()
    }
    symbol_heatmap = pd.DataFrame(symbol_series_by_window).sort_index().T.reindex(heatmap_windows)
    if symbol_heatmap.columns.size > 0:
        benchmark_heatmap_matrices[symbol] = symbol_heatmap
        benchmark_daily_mean[symbol] = symbol_heatmap.mean(axis=0, skipna=True).dropna().sort_index()

heatmap_colorscale = [
    (0.0, "#b2182b"),
    (0.5, "#f7f7f7"),
    (1.0, "#1a9850"),
]

def collect_heatmap_abs_values(matrices):
    finite_abs_chunks = []
    for matrix in matrices:
        if matrix is None or matrix.empty:
            continue
        values = matrix.to_numpy(dtype=float)
        finite_abs_values = np.abs(values[np.isfinite(values)])
        if finite_abs_values.size > 0:
            finite_abs_chunks.append(finite_abs_values)
    if finite_abs_chunks:
        return np.concatenate(finite_abs_chunks)
    return np.array([], dtype=float)

def build_heatmap_time_range_buttons(global_start, global_end, axis_count=1):
    def make_heatmap_time_range(*, months=None, years=None):
        if months is not None:
            start = max(global_start, global_end - pd.DateOffset(months=months))
        elif years is not None:
            start = max(global_start, global_end - pd.DateOffset(years=years))
        else:
            start = global_start
        layout_updates = {}
        for axis_idx in range(1, axis_count + 1):
            axis_name = "xaxis" if axis_idx == 1 else f"xaxis{axis_idx}"
            layout_updates[f"{axis_name}.range"] = [start, global_end]
        return dict(method="relayout", args=[layout_updates])

    return [
        dict(label="10 Years", **make_heatmap_time_range(years=10)),
        dict(label="5 Years", **make_heatmap_time_range(years=5)),
        dict(label="3 Years", **make_heatmap_time_range(years=3)),
        dict(label="1 Year", **make_heatmap_time_range(years=1)),
        dict(label="6 Months", **make_heatmap_time_range(months=6)),
        dict(label="3 Months", **make_heatmap_time_range(months=3)),
        dict(label="1 Month", **make_heatmap_time_range(months=1)),
    ], max(global_start, global_end - pd.DateOffset(months=6))

def build_secondary_axis_range(series_collection):
    finite_chunks = []
    for series in series_collection:
        if series is None or len(series) == 0:
            continue
        values = np.asarray(series, dtype=float)
        finite_values = values[np.isfinite(values)]
        if finite_values.size > 0:
            finite_chunks.append(finite_values)
    if not finite_chunks:
        return [-1.0, 1.0]
    combined_values = np.concatenate(finite_chunks)
    lower_bound = min(float(combined_values.min()), 0.0)
    upper_bound = max(float(combined_values.max()), 0.0)
    span = upper_bound - lower_bound
    padding = max(span * 0.08, 0.25)
    return [lower_bound - padding, upper_bound + padding]

asset_heatmap_abs_values = collect_heatmap_abs_values([heatmap_matrix])
has_asset_values = asset_heatmap_abs_values.size > 0
asset_observed_max_abs_zscore = float(asset_heatmap_abs_values.max()) if has_asset_values else 1.0
asset_color_scale_cap = float(np.nanpercentile(asset_heatmap_abs_values, 97.5)) if has_asset_values else 1.0
asset_color_scale_cap = max(asset_color_scale_cap, 1.0)

benchmark_abs_values = collect_heatmap_abs_values(list(benchmark_heatmap_matrices.values()))
has_benchmark_values = benchmark_abs_values.size > 0
benchmark_observed_max_abs_zscore = float(benchmark_abs_values.max()) if has_benchmark_values else 1.0
benchmark_color_scale_cap = float(np.nanpercentile(benchmark_abs_values, 97.5)) if has_benchmark_values else 1.0
benchmark_color_scale_cap = max(benchmark_color_scale_cap, 1.0)

benchmark_order_heatmap = [
    symbol for symbol in benchmark_plot_payload["benchmark_order"] if symbol in benchmark_heatmap_matrices
]
default_benchmark_heatmap = benchmark_plot_payload["default_benchmark"]
if default_benchmark_heatmap not in benchmark_order_heatmap:
    default_benchmark_heatmap = benchmark_order_heatmap[0] if benchmark_order_heatmap else None

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.5, 0.5],
    specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    subplot_titles=(
        "Asset Sharpe Z-Score by Rolling Window",
        "Benchmark Sharpe Spread Z-Score by Rolling Window",
    ),
)

constant_trace_indices = []

fig.add_trace(
    go.Heatmap(
        z=heatmap_matrix.values,
        x=heatmap_matrix.columns,
        y=heatmap_matrix.index,
        colorscale=heatmap_colorscale,
        zmid=0,
        zmin=-asset_color_scale_cap,
        zmax=asset_color_scale_cap,
        colorbar=dict(title="Sharpe Z-Score", x=1.11, y=0.81, len=0.34),
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Window: %{y} day(s)<br>"
            "Sharpe Z-Score: %{z:.2f}<extra></extra>"
        ),
        showscale=True,
        name="Asset Heatmap",
    ),
    row=1,
    col=1,
)
constant_trace_indices.append(len(fig.data) - 1)

asset_line_x = asset_daily_mean.index if not asset_daily_mean.empty else heatmap_matrix.columns
if not asset_daily_mean.empty:
    fig.add_trace(
        go.Scatter(
            x=asset_daily_mean.index,
            y=asset_daily_mean,
            mode="lines",
            name="Asset Mean Across Windows",
            line=dict(color="#38bdf8", width=2.2, dash="dash"),
            hovertemplate="Date: %{x|%Y-%m-%d}<br>Mean Sharpe Z-Score: %{y:.2f}<extra></extra>",
        ),
        row=1,
        col=1,
        secondary_y=True,
    )
    constant_trace_indices.append(len(fig.data) - 1)

if len(asset_line_x) > 0:
    fig.add_trace(
        go.Scatter(
            x=asset_line_x,
            y=np.zeros(len(asset_line_x)),
            mode="lines",
            name="Asset Zero Line",
            line=dict(color="rgba(226, 232, 240, 0.60)", width=1.4),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=1,
        secondary_y=True,
    )
    constant_trace_indices.append(len(fig.data) - 1)

benchmark_trace_bounds = {}
for symbol in benchmark_order_heatmap:
    symbol_matrix = benchmark_heatmap_matrices[symbol]
    visible = symbol == default_benchmark_heatmap
    fig.add_trace(
        go.Heatmap(
            z=symbol_matrix.values,
            x=symbol_matrix.columns,
            y=symbol_matrix.index,
            colorscale=heatmap_colorscale,
            zmid=0,
            zmin=-benchmark_color_scale_cap,
            zmax=benchmark_color_scale_cap,
            colorbar=dict(title="Sharpe Spread Z-Score", x=1.11, y=0.19, len=0.34),
            hovertemplate=(
                "Benchmark: " + symbol + "<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Window: %{y} day(s)<br>"
                "Sharpe Spread Z-Score: %{z:.2f}<extra></extra>"
            ),
            visible=visible,
            showscale=True,
            name=f"{symbol} Heatmap",
        ),
        row=2,
        col=1,
    )
    heatmap_trace_idx = len(fig.data) - 1

    symbol_mean = benchmark_daily_mean.get(symbol, pd.Series(dtype=float))
    fig.add_trace(
        go.Scatter(
            x=symbol_mean.index,
            y=symbol_mean,
            mode="lines",
            name="Benchmark Mean Across Windows",
            line=dict(color="#38bdf8", width=2.2, dash="dash"),
            hovertemplate=(
                "Benchmark: " + symbol + "<br>"
                "Date: %{x|%Y-%m-%d}<br>"
                "Mean Sharpe Spread Z-Score: %{y:.2f}<extra></extra>"
            ),
            visible=visible,
            showlegend=False,
        ),
        row=2,
        col=1,
        secondary_y=True,
    )
    mean_trace_idx = len(fig.data) - 1
    benchmark_trace_bounds[symbol] = (heatmap_trace_idx, mean_trace_idx)

benchmark_line_x = max(
    [matrix.columns for matrix in benchmark_heatmap_matrices.values() if matrix.columns.size > 0],
    key=len,
    default=pd.Index([]),
)
if len(benchmark_line_x) > 0:
    fig.add_trace(
        go.Scatter(
            x=benchmark_line_x,
            y=np.zeros(len(benchmark_line_x)),
            mode="lines",
            name="Benchmark Zero Line",
            line=dict(color="rgba(226, 232, 240, 0.60)", width=1.4),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=2,
        col=1,
        secondary_y=True,
    )
    constant_trace_indices.append(len(fig.data) - 1)

figure_title = f"{ticker_str} Rolling Sharpe Z-Score Heatmaps (1-200 Day Windows)"
if default_benchmark_heatmap is not None:
    figure_title += f" | Benchmark: {default_benchmark_heatmap}"

fig.update_layout(
    title=figure_title,
    template="plotly_dark",
    height=1180,
    margin=dict(l=70, r=150, t=95, b=55),
    legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="right", x=1.0),
)

fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text="Rolling Window (Days)", autorange="reversed", row=1, col=1)
fig.update_yaxes(
    title_text="Mean Sharpe Z-Score",
    range=build_secondary_axis_range([asset_daily_mean]),
    showgrid=False,
    zeroline=False,
    row=1,
    col=1,
    secondary_y=True,
)
fig.update_yaxes(title_text="Rolling Window (Days)", autorange="reversed", row=2, col=1)
fig.update_yaxes(
    title_text="Mean Sharpe Spread Z-Score",
    range=build_secondary_axis_range(benchmark_daily_mean.values()),
    showgrid=False,
    zeroline=False,
    row=2,
    col=1,
    secondary_y=True,
)

updatemenus = []

date_ranges = []
if heatmap_matrix.columns.size > 0:
    date_ranges.append((heatmap_matrix.columns.min(), heatmap_matrix.columns.max()))
date_ranges.extend(
    (matrix.columns.min(), matrix.columns.max())
    for matrix in benchmark_heatmap_matrices.values()
    if matrix.columns.size > 0
)

if benchmark_order_heatmap:
    total_traces = len(fig.data)
    benchmark_buttons = []
    for symbol in benchmark_order_heatmap:
        visibility = [False] * total_traces
        for trace_idx in constant_trace_indices:
            visibility[trace_idx] = True
        benchmark_heatmap_idx, benchmark_mean_idx = benchmark_trace_bounds[symbol]
        visibility[benchmark_heatmap_idx] = True
        visibility[benchmark_mean_idx] = True
        benchmark_buttons.append(
            dict(
                label=symbol,
                method="update",
                args=[
                    {"visible": visibility},
                    {
                        "title": (
                            f"{ticker_str} Rolling Sharpe Z-Score Heatmaps (1-200 Day Windows) | Benchmark: {symbol}"
                        )
                    },
                ],
            )
        )
    updatemenus.append(
        dict(
            buttons=benchmark_buttons,
            direction="down",
            showactive=True,
            x=0.01,
            y=1.12,
            xanchor="left",
            yanchor="top",
            active=benchmark_order_heatmap.index(default_benchmark_heatmap),
        )
    )

if date_ranges:
    global_start = min(start for start, _ in date_ranges)
    global_end = max(end for _, end in date_ranges)
    heatmap_time_range_buttons, default_start = build_heatmap_time_range_buttons(global_start, global_end, axis_count=2)
    fig.update_xaxes(range=[default_start, global_end], row=1, col=1)
    fig.update_xaxes(range=[default_start, global_end], row=2, col=1)
    updatemenus.append(
        dict(
            buttons=heatmap_time_range_buttons,
            direction="down",
            showactive=True,
            x=0.18,
            y=1.12,
            xanchor="left",
            yanchor="top",
            active=4,
        )
    )

if updatemenus:
    fig.update_layout(updatemenus=updatemenus)

if asset_observed_max_abs_zscore > asset_color_scale_cap:
    fig.add_annotation(
        text=(
            f"Asset heatmap color scale capped at +/-{asset_color_scale_cap:.2f}; observed max is "
            f"{asset_observed_max_abs_zscore:.2f}."
        ),
        xref="paper",
        yref="paper",
        x=1,
        y=1.17,
        xanchor="right",
        yanchor="top",
        showarrow=False,
        font=dict(size=11, color="rgba(220, 220, 220, 0.85)"),
    )

if 1 in heatmap_matrix.index and heatmap_matrix.loc[1].isna().all():
    fig.add_annotation(
        text="Asset 1-day window is blank because rolling Sharpe requires at least two return observations.",
        xref="paper",
        yref="paper",
        x=1,
        y=1.12,
        xanchor="right",
        yanchor="top",
        showarrow=False,
        font=dict(size=11, color="rgba(220, 220, 220, 0.85)"),
    )

if benchmark_order_heatmap and benchmark_observed_max_abs_zscore > benchmark_color_scale_cap:
    fig.add_annotation(
        text=(
            f"Benchmark heatmap color scale capped at +/-{benchmark_color_scale_cap:.2f}; observed max is "
            f"{benchmark_observed_max_abs_zscore:.2f}."
        ),
        xref="paper",
        yref="paper",
        x=1,
        y=0.46,
        xanchor="right",
        yanchor="top",
        showarrow=False,
        font=dict(size=11, color="rgba(220, 220, 220, 0.85)"),
    )

if benchmark_order_heatmap:
    default_benchmark_matrix = benchmark_heatmap_matrices[default_benchmark_heatmap]
    if 1 in default_benchmark_matrix.index and default_benchmark_matrix.loc[1].isna().all():
        fig.add_annotation(
            text="Benchmark 1-day window is blank because rolling Sharpe requires at least two return observations.",
            xref="paper",
            yref="paper",
            x=1,
            y=0.41,
            xanchor="right",
            yanchor="top",
            showarrow=False,
            font=dict(size=11, color="rgba(220, 220, 220, 0.85)"),
        )
else:
    fig.add_annotation(
        text="No benchmark Sharpe spread z-score heatmap data available.",
        xref="paper",
        yref="paper",
        x=0.5,
        y=0.25,
        showarrow=False,
        font=dict(size=12, color="rgba(220, 220, 220, 0.85)"),
    )

show_plotly_figure(fig)


In [ ]:
# Block 10: visualize monthly, weekly, and daily seasonality patterns

#Seasonality: Monthly, Weekly, and Daily Returns
#SEASONALITY: Monthly Seasonality
fig_ticker_Seasonality_Monthly = barChartPlotter.create_seasonality_fig(ticker_monthly_returns, f'Monthly Seasonality: {ticker_str}', 'monthly')
show_plotly_figure(fig_ticker_Seasonality_Monthly)

#SEASONALITY: Weekly Seasonality
fig_ticker_Seasonality_weekly = barChartPlotter.create_seasonality_fig(ticker_weekly_returns, f'Weekly Seasonality: {ticker_str}', 'weekly')
show_plotly_figure(fig_ticker_Seasonality_weekly)

#SEASONALITY: Daily Seasonality
fig_ticker_Seasonality_daily = barChartPlotter.create_seasonality_fig(ticker_daily_returns, f'Daily Seasonality: {ticker_str}', 'daily')
show_plotly_figure(fig_ticker_Seasonality_daily)

In [ ]:
# Block 11: compute Sharpe/Sortino ratios and spreads

import importlib
import Quantapp.analytics.risk_relative_analytics as risk_relative_analytics_module

# Refresh the analytics class in case the notebook kernel still holds an older signature.
importlib.reload(risk_relative_analytics_module)
risk_relative_analytics = risk_relative_analytics_module.RiskRelativeAnalytics()

asset_close = ticker['Close']

risk_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_close,
    time_frame_map=time_frame_map,
    benchmark_data=benchmark_data,
    risk_free_rate=risk_free_daily_rate,
)

asset_sharpe_map = risk_context['asset_sharpe_map']
asset_component_map = risk_context['asset_component_map']
asset_sortino_map = risk_context['asset_sortino_map']
asset_sharpe_sortino_spread_map = risk_context['asset_sharpe_sortino_spread_map']

benchmark_metrics = risk_context['benchmark_metrics']
benchmark_order = risk_context['benchmark_order']
default_benchmark = risk_context['default_benchmark']
spread_plot_data = risk_context['spread_plot_data']
term_config_map = risk_context['term_config_map']


In [ ]:
# Block 12: plot Sharpe & Sortino efficiency with dropdown controls

long_window = time_frame_map.get('long')
default_term_label = f"{long_window}-day" if long_window is not None else max(term_config_map, key=lambda label: int(str(label).split('-')[0]))

fig = lineChartPlotter.plot_sharpe_sortino_comparison(
    term_config_map=term_config_map,
    ticker_label=ticker_str,
    default_label=default_term_label,
)
show_plotly_figure(fig)


In [ ]:
# Block 13: plot rolling correlation of the asset versus benchmarks

asset_daily_returns = ticker['Close'].pct_change(fill_method=None)
correlation_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
correlation_benchmark_order = benchmark_order if benchmark_order else list(benchmark_data.keys())
rolling_correlation_map = {}

for term in correlation_term_order:
    window = int(time_frame_map[term])
    term_series_map = {}

    for symbol in correlation_benchmark_order:
        benchmark_frame = benchmark_data.get(symbol)
        if benchmark_frame is None or 'Close' not in benchmark_frame:
            continue

        benchmark_daily_returns = benchmark_frame['Close'].pct_change(fill_method=None)
        aligned_returns = pd.concat(
            [
                asset_daily_returns.rename('asset'),
                benchmark_daily_returns.rename(symbol),
            ],
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns['asset'].rolling(window).corr(aligned_returns[symbol]).dropna()
        if rolling_correlation_series.empty:
            continue

        term_series_map[symbol] = rolling_correlation_series

    if term_series_map:
        rolling_correlation_map[term] = term_series_map

if not rolling_correlation_map:
    print('No rolling correlation data available for benchmark comparison.')
else:
    rolling_correlation_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    benchmark_colors = {
        symbol: rolling_correlation_palette[idx % len(rolling_correlation_palette)]
        for idx, symbol in enumerate(correlation_benchmark_order)
    }
    correlation_rows = list(rolling_correlation_map.keys())
    rolling_correlation_fig = make_subplots(
        rows=len(correlation_rows),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=[f"{time_frame_map[term]}-Day Rolling Correlation" for term in correlation_rows],
    )

    for annotation in rolling_correlation_fig.layout.annotations:
        annotation.font = dict(size=11, color='rgba(220, 220, 220, 0.90)')

    date_ranges = []
    for row_idx, term in enumerate(correlation_rows, start=1):
        row_series_map = rolling_correlation_map[term]
        row_values = []

        for symbol_idx, (symbol, correlation_series) in enumerate(row_series_map.items()):
            row_values.append(correlation_series)
            date_ranges.append((correlation_series.index.min(), correlation_series.index.max()))
            rolling_correlation_fig.add_trace(
                go.Scatter(
                    x=correlation_series.index,
                    y=correlation_series,
                    mode='lines',
                    name=symbol,
                    legendgroup=symbol,
                    line=dict(color=benchmark_colors[symbol], width=2),
                    hovertemplate=f"Benchmark: {symbol}<br>Date: %{{x|%Y-%m-%d}}<br>Rolling Correlation: %{{y:.2f}}<extra></extra>",
                    showlegend=row_idx == 1,
                ),
                row=row_idx,
                col=1,
            )
            rolling_correlation_fig.add_hline(
                y=correlation_series.mean(),
                line_dash='dot',
                line_color=benchmark_colors[symbol],
                line_width=1,
                opacity=0.7,
                row=row_idx,
                col=1,
            )
            mean_annotation_x = max(0.18, 0.985 - (symbol_idx * 0.18))
            rolling_correlation_fig.add_annotation(
                x=mean_annotation_x,
                y=correlation_series.mean(),
                xref='x domain',
                yref='y',
                text=f"{symbol} Mean {correlation_series.mean():.2f}",
                showarrow=False,
                xanchor='right',
                yanchor='bottom',
                font=dict(size=10, color=benchmark_colors[symbol]),
                row=row_idx,
                col=1,
            )

        rolling_correlation_fig.add_hline(
            y=0,
            line_dash='dash',
            line_color='rgba(180, 180, 180, 0.65)',
            line_width=1,
            opacity=0.8,
            row=row_idx,
            col=1,
        )
        rolling_correlation_fig.add_annotation(
            x=0.985,
            y=0,
            xref='x domain',
            yref='y',
            text='Zero',
            showarrow=False,
            xanchor='right',
            yanchor='top',
            font=dict(size=10, color='rgba(200, 200, 200, 0.85)'),
            row=row_idx,
            col=1,
        )

        if row_values:
            row_frame = pd.concat(row_values, axis=1)
            row_min = min(row_frame.min().min(), 0)
            row_max = max(row_frame.max().max(), 0)
            row_span = row_max - row_min
            row_padding = max(row_span * 0.08, 0.03) if pd.notna(row_span) else 0.03
            rolling_correlation_fig.update_yaxes(
                title_text='Correlation',
                range=[row_min - row_padding, row_max + row_padding],
                row=row_idx,
                col=1,
            )

    if date_ranges:
        global_start = min(start for start, _ in date_ranges)
        global_end = max(end for _, end in date_ranges)
        default_start = max(global_start, global_end - pd.DateOffset(years=3))
        for axis_idx in range(1, len(correlation_rows) + 1):
            axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
            rolling_correlation_fig.layout[axis_name].update(range=[default_start, global_end])
        time_range_menu = dict(
            buttons=build_time_range_buttons(global_start, global_end, axis_count=len(correlation_rows)),
            direction='down',
            showactive=True,
            x=0.01,
            y=1.12,
            xanchor='left',
            yanchor='top',
            active=2,
        )
    else:
        time_range_menu = None

    rolling_correlation_fig.update_layout(
        title=f'{ticker_str} Rolling Correlation vs Benchmarks',
        template='plotly_dark',
        height=max(760, 240 * len(correlation_rows) + 140),
        margin=dict(l=50, r=40, t=90, b=40),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        updatemenus=[time_range_menu] if time_range_menu is not None else [],
    )
    rolling_correlation_fig.update_xaxes(title_text='Date', row=len(correlation_rows), col=1)
    show_plotly_figure(rolling_correlation_fig)


In [ ]:
# Block 9: render interactive momentum z-score comparisons

window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_analytics.momentum_zscore_map(
    ticker['Close'],
    window_pairs=window_pairs,
)

fig = lineChartPlotter.plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="200 vs 400",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.update_layout(height=850)
show_plotly_figure(fig)


In [ ]:
# Block 14: combine risk-adjusted return and benchmark plots
# Requires the current kernel session to have fresh outputs from Blocks 2, 5, and 11.

if benchmark_order:
    benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
        asset_sharpe_map=asset_sharpe_map,
        asset_component_map=asset_component_map,
        benchmark_metrics=benchmark_metrics,
        spread_plot_data=spread_plot_data,
        time_frame_map=time_frame_map,
    )

    default_term = 'long' if 'long' in time_frame_map else max(time_frame_map, key=time_frame_map.get)

    summary_fig = lineChartPlotter.plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=benchmark_plot_payload['summary_zscore_map'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_term=default_term,
    )
    show_plotly_figure(summary_fig)

    detail_fig = lineChartPlotter.plot_benchmark_zscore_detail(
        detail_zscore_map=benchmark_plot_payload['detail_zscore_map'],
        benchmark_order=benchmark_plot_payload['benchmark_order'],
        time_frame_map=time_frame_map,
        ticker_label=ticker_str,
        default_benchmark=benchmark_plot_payload['default_benchmark'],
        default_term=default_term,
    )
    show_plotly_figure(detail_fig)
else:
    print('No benchmark data available for benchmark comparison plots.')


In [ ]:
# Block 15: factor attribution moved to its own file
print("Factor attribution now lives in Factor Analysis.ipynb. Open that notebook to run the FF5 and idiosyncratic-risk workflow.")
